In [3]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patches as mpatches
from matplotlib.colors import LinearSegmentedColormap, to_hex
import seaborn as sns
import scanpy as sc
import anndata as ad
from scipy import sparse
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set scanpy settings
sc.settings.verbosity = 3  # verbosity level

print("Libraries imported successfully!")

Libraries imported successfully!


In [4]:
adata_1 = ad.read_h5ad('combined_adata_first_half.h5ad')
print(adata_1)

AnnData object with n_obs × n_vars = 487905 × 33538
    obs: 'barcodes', 'sample'


In [5]:
adata_2 = ad.read_h5ad('combined_adata_second_half.h5ad')
print(adata_2)

AnnData object with n_obs × n_vars = 338345 × 33538
    obs: 'barcodes', 'sample'


In [1]:
import scanpy as sc
import scipy.sparse as sp

input_h5ad = "colon_adata_clustered.h5ad"
output_h5ad = "colon_adata_for_seurat.h5ad"

adata = sc.read_h5ad(input_h5ad)

# If you know your raw counts are in adata.raw, set this to True and use adata.raw.X instead.
USE_RAW_COUNTS = False

if USE_RAW_COUNTS and adata.raw is not None:
    X = adata.raw.X
    var_names = adata.raw.var_names
else:
    X = adata.X
    var_names = adata.var_names

# Keep only what the R workflow needs
keep_obs = ["fov", "cell_id", "nCount_RNA", "nFeature_RNA"]
keep_obs = [c for c in keep_obs if c in adata.obs.columns]
adata.obs = adata.obs[keep_obs].copy()

# Drop large/unneeded containers
adata.obsm.clear()
adata.obsp.clear()
adata.uns = {}
adata.layers.clear()
adata.raw = None

# Make sure names are valid and unique
adata.obs_names_make_unique()
adata.var_names = var_names
adata.var_names_make_unique()

# Keep expression matrix in sparse CSR float32
if not sp.issparse(X):
    X = sp.csr_matrix(X)
adata.X = X.astype("float32")

# Ensure orientation is cells x genes for Seurat/R
if adata.X.shape[0] != adata.n_obs:
    adata.X = adata.X.T.tocsr().astype("float32")

adata.write_h5ad(output_h5ad, compression="gzip")
print(f"Wrote {output_h5ad}")
print(adata)

Wrote colon_adata_for_seurat.h5ad
AnnData object with n_obs × n_vars = 424423 × 3000
    obs: 'fov', 'cell_id', 'nCount_RNA', 'nFeature_RNA'
    var: 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'mean', 'std'
    varm: 'PCs'


In [2]:
import scanpy as sc
import scipy.sparse as sp
from scipy.io import mmwrite
from pathlib import Path

input_h5ad = "colon_adata_for_seurat.h5ad"

out_counts = Path("colon_adata_for_seurat_counts.mtx")
out_cells = Path("colon_adata_for_seurat_cells.tsv")
out_genes = Path("colon_adata_for_seurat_genes.tsv")
out_obs = Path("colon_adata_for_seurat_obs.csv")

adata = sc.read_h5ad(input_h5ad)

X = adata.X
if not sp.issparse(X):
    X = sp.csr_matrix(X)
X = X.astype("float32")

# export in genes x cells order for R / Seurat
X_export = X.T.tocoo()
mmwrite(str(out_counts), X_export)

with out_cells.open("w", encoding="utf-8") as handle:
    for cell_name in map(str, adata.obs_names):
        handle.write(f"{cell_name}\n")

with out_genes.open("w", encoding="utf-8") as handle:
    for gene_name in map(str, adata.var_names):
        handle.write(f"{gene_name}\n")

obs = adata.obs.copy()
obs = obs.reset_index(drop=False)
obs.to_csv(out_obs, index=False)

print(f"Wrote {out_counts}")
print(f"Wrote {out_cells}")
print(f"Wrote {out_genes}")
print(f"Wrote {out_obs}")
print(f"Matrix exported as genes x cells: {X_export.shape[0]} x {X_export.shape[1]}")

Wrote colon_adata_for_seurat_counts.mtx
Wrote colon_adata_for_seurat_cells.tsv
Wrote colon_adata_for_seurat_genes.tsv
Wrote colon_adata_for_seurat_obs.csv
Matrix exported as genes x cells: 3000 x 424423
